# Values, Types, Operators & Expressions

## What's covered

- The big mental shift — **variables are names that refer to objects**, not boxes that contain values
- The object model — everything is an object, including types, functions, modules
- The built-in types — `int`, `float`, `bool`, `str`, `bytes`, `None`, `list`, `tuple`, `dict`, `set`
- Numeric types — arbitrary-precision `int`, IEEE-754 `float`, fixed-point `Decimal`, rational `Fraction`, `complex`
- `bool` is a subclass of `int` — and what that implies
- The string type — Unicode by default, immutable, `bytes` is the binary cousin
- String literals: regular, raw (`r"..."`), f-strings, byte strings
- `None` — the singleton you compare with `is None`, not `== None`
- Arithmetic operators including integer division `//`, modulo `%`, exponentiation `**`
- Comparison operators — and the chain shorthand `a < b < c`
- Boolean operators — `and`, `or`, `not`, and their short-circuit behaviour
- Bitwise operators
- Identity (`is`) vs equality (`==`) — when each fits
- Truthiness — what Python considers "false-y"
- Mutable vs immutable — the property that decides aliasing behaviour
- Implicit string concatenation — the tiny syntactic gotcha
- Type conversion — explicit (`int(x)`) vs the lack of implicit
- The walrus operator `:=`
- Operator precedence — the table you bookmark

## The big shift — variables are names

Coming from a statically typed language, the natural mental model is "a variable is a box that contains a value." In Python, that model is wrong and it will produce subtle bugs.

The right mental model: **a variable is a name that refers to an object.** Objects live on the heap. Assignment binds a name to an object — it does not copy the object.

```python
a = [1, 2, 3]
b = a            # b refers to the SAME list object as a
b.append(4)
print(a)         # [1, 2, 3, 4] — a sees the change too
```

This matters because mutation through one name is visible through every other name pointing at the same object. We will see this constantly with lists, dicts, and sets — and never with `int`, `float`, `str`, `tuple` (because those are immutable; "mutating" them just creates a new object).

In [ ]:
# Variables are names; assignment binds.
a = [1, 2, 3]
b = a            # b and a refer to the same list
b.append(4)
print(a)         # [1, 2, 3, 4] — aliasing
print(b)         # [1, 2, 3, 4]
print(a is b)    # True — they ARE the same object

# Reassigning a name does not mutate the object — it points the name elsewhere
b = [99, 100]
print(a)         # [1, 2, 3, 4] — a still points at the original
print(b)         # [99, 100] — b now points at a new list

## Everything is an object

Every value in Python — `42`, `"hello"`, a list, a function you defined, the `int` type itself, the module you imported — is an object. Objects have:

- An **identity** — a unique number you can see with `id(x)`. Effectively the memory address in CPython.
- A **type** — visible with `type(x)`. Lives in the object's header.
- A **value** — the actual data.

Functions, classes, and modules are objects too. You can pass a function as an argument, assign a class to a variable, store modules in a dictionary. This makes a lot of Python idioms feel natural — decorators (notebook 07) are just functions taking other functions; metaclasses are classes whose instances are classes themselves.

In [ ]:
# Every value has an identity, a type, and a value
x = 42
print(id(x))     # some integer — the object's identity
print(type(x))   # <class 'int'>

# Functions are objects
def greet(name):
    return f"hello, {name}"

print(type(greet))               # <class 'function'>
print(greet.__name__)            # 'greet'
my_alias = greet                 # functions can be assigned to variables
print(my_alias("ganesh"))        # hello, ganesh

# Types themselves are objects (instances of `type`)
print(type(int))                 # <class 'type'>
print(int)                       # <class 'int'> — the type object itself

# A list of types — yes, perfectly valid
numeric_types = [int, float, complex]
for t in numeric_types:
    print(t, t(0))

## The built-in types — a quick map

The types you'll meet in the first hour, organized by mutability and "shape":

| Category | Type | Mutable? | Literal |
|---|---|---|---|
| Numeric | `int`        | no  | `42`, `0x2a`, `0b101010` |
| Numeric | `float`      | no  | `3.14`, `1e6`, `inf`, `nan` |
| Numeric | `complex`    | no  | `3+4j` |
| Numeric | `bool`       | no  | `True`, `False` |
| Text    | `str`        | no  | `"hello"`, `'hello'`, `f"hi {name}"` |
| Binary  | `bytes`      | no  | `b"hello"` |
| Binary  | `bytearray`  | yes | `bytearray(b"hello")` |
| None    | `NoneType`   | n/a | `None` |
| Sequence| `list`       | yes | `[1, 2, 3]` |
| Sequence| `tuple`      | no  | `(1, 2, 3)`, `1, 2, 3` |
| Sequence| `range`      | no  | `range(10)` |
| Mapping | `dict`       | yes | `{"a": 1}` |
| Set     | `set`        | yes | `{1, 2, 3}`, `set()` |
| Set     | `frozenset`  | no  | `frozenset([1, 2, 3])` |

Notebook 04 covers the collection types (`list`, `tuple`, `dict`, `set`) properly. This notebook focuses on the scalar/atomic types.

## Numeric types — int, float, Decimal, Fraction, complex

**`int` is arbitrary-precision.** Unlike C, Java, or even most modern languages, Python integers have no `int.MaxValue`. They grow as large as you have memory for. `10 ** 100` is a number, not an overflow.

**`float` is IEEE-754 double.** Same caveats as every other language — `0.1 + 0.2 != 0.3`, special values like `inf` and `nan`. For exact decimal arithmetic, reach for `decimal.Decimal`. For exact fractions, `fractions.Fraction`.

**`bool` is a subclass of `int`.** `True == 1` and `False == 0`, and `True + 1 == 2`. This sounds weird, and it occasionally is — `sum([True, True, False, True]) == 3` is technically correct Python.

**`complex` exists in the standard language.** `3 + 4j` is a literal. Not used heavily outside scientific work; mentioned for completeness.

In [ ]:
# int — arbitrary precision
big = 2 ** 1000
print(len(str(big)))          # 302 — yes, a 302-digit number, no overflow

# Different bases for int literals
print(0b1010)                 # 10  (binary)
print(0o17)                   # 15  (octal)
print(0xff)                   # 255 (hex)

# Underscores for readability in numeric literals (3.6+)
million = 1_000_000
print(million)                # 1000000

# float — IEEE-754 double
print(0.1 + 0.2)              # 0.30000000000000004 — same as every other language
print(1e308 * 10)             # inf
print(float("nan") == float("nan"))  # False — NaN is never equal to itself

# Decimal — exact decimal arithmetic
from decimal import Decimal
print(Decimal("0.1") + Decimal("0.2"))   # 0.3 — exact

# Fraction — exact rational
from fractions import Fraction
print(Fraction(1, 3) + Fraction(1, 6))   # 1/2

# bool is an int — really
print(True + True + False + True)        # 3
print(isinstance(True, int))             # True

## Strings — Unicode by default, immutable

`str` is Python's text type. It is **Unicode** by default — a `str` is a sequence of Unicode code points, not bytes. The encoding question (UTF-8, UTF-16, ASCII) only comes up at I/O boundaries, when you encode `str` to `bytes` to write to a file or socket, or decode `bytes` back to `str` when reading.

`str` is **immutable**. Operations like `.upper()`, `.replace()`, `.strip()` return new strings. You cannot modify a string in place.

There are several literal forms — same string, different parsing rules:

| Form | Example | What it does |
|---|---|---|
| Regular | `"hello"` or `'hello'` | normal string, backslash escapes |
| Raw | `r"path\to\file"` | backslashes are literal — no escapes |
| f-string | `f"hi {name}"` | values spliced in via `{...}` |
| Triple-quoted | `"""multi\nline"""` | newlines allowed inside |
| Bytes | `b"hello"` | binary `bytes` type, not `str` |
| Combined | `rf"path\to\{name}"` | raw and f-string at once |

In [ ]:
# Regular string with backslash escapes
print("line one\nline two")

# Raw string — backslashes are literal
print(r"C:\Users\ganesh\file.txt")

# f-string with expressions inside {...}
name = "ganesh"
score = 92.345
print(f"hello, {name}")
print(f"score = {score:.2f}")           # format spec after colon
print(f"name length = {len(name)}")     # any expression works
print(f"{name=}")                       # debug form: prints "name='ganesh'" (3.8+)

# Triple-quoted — multi-line, includes embedded quotes freely
sql = """SELECT id, name
FROM users
WHERE name = 'ganesh'"""
print(sql)

# bytes vs str — DIFFERENT TYPES, can't be mixed
b = b"hello"
s = "hello"
print(type(b), type(s))                 # <class 'bytes'> <class 'str'>
# print(b + s)                          # TypeError
print(b.decode("utf-8") == s)           # True — decoded bytes equal the str

## `None` — and the `is None` rule

`None` is Python's null. It's a singleton — there is exactly one `None` object in the entire interpreter. That singleton-ness is why you should always compare with `is None`, never `== None`:

```python
if x is None: ...        # correct
if x == None: ...        # works but worse — invokes __eq__, can be overridden
```

`is None` checks object identity (same object?), `== None` checks equality (does `__eq__` return True?). Some objects override `__eq__` in ways that make them compare equal to `None`. `is None` is faster, clearer, and immune to that.

The same rule applies to `True` and `False` — `if x is True` is rarely what you want (use `if x:` for truthiness), but if you really mean "literally the `True` singleton," `is True` is the right form.

In [ ]:
x = None

# The right form
if x is None:
    print("absent")

# Why `==` is the wrong tool
class WeirdEqual:
    def __eq__(self, other):
        return True              # equal to literally everything

w = WeirdEqual()
print(w == None)                 # True — surprise
print(w is None)                 # False — correctly distinguishes

## Arithmetic operators

The basics behave as you'd expect. Three things worth flagging:

| Operator | Meaning | Note |
|---|---|---|
| `+`, `-`, `*` | usual | also defined on strings (`"ab" * 3`) and lists |
| `/` | **true division** — always returns `float` | `7 / 2 == 3.5`, even on ints |
| `//` | **floor division** — rounds toward negative infinity | `7 // 2 == 3`, `-7 // 2 == -4` |
| `%` | modulo — same sign as divisor | `-7 % 3 == 2` |
| `**` | exponentiation | `2 ** 10 == 1024` |
| `divmod(a, b)` | returns `(a // b, a % b)` | one call, both pieces |

The thing to remember: **`/` always returns a float** in Python 3 — even when both operands are integers. Use `//` when you want integer division. This is one of the biggest changes from Python 2.

In [ ]:
# True vs floor division
print(7 / 2)        # 3.5 — float
print(7 // 2)       # 3   — int

# Floor division rounds toward negative infinity (not toward zero!)
print(-7 // 2)      # -4 — not -3
print(7 % -3)       # -2 — modulo's sign matches the divisor

# Exponentiation
print(2 ** 10)      # 1024
print(2 ** 0.5)     # 1.4142...  — fractional exponents work, gives float
print(10 ** -2)     # 0.01

# divmod — quotient and remainder in one call
print(divmod(17, 5))   # (3, 2)

# Operators on strings
print("ab" * 3)     # "ababab"
print("hello, " + "world")

# Augmented assignment — n += 1 is n = n + 1, but with one extra trick for mutables
xs = [1, 2, 3]
xs += [4, 5]        # extends in place for lists — same as xs.extend([4, 5])
print(xs)

## Comparison operators and chain comparisons

`==`, `!=`, `<`, `<=`, `>`, `>=` work as expected. The Python-specific feature: **comparisons chain**. `a < b < c` is shorthand for `a < b and b < c`, with `b` evaluated only once.

```python
0 <= x < 100        # natural — x is in [0, 100)
3 < x < 10 < y < 20 # works, but unreadable past two operators
```

The chain form is the right choice for the common case "is `x` in some range." It is also the only place in Python where `and` is implicit.

In [ ]:
x = 42
print(0 <= x < 100)              # True — chain
print(0 <= x and x < 100)        # True — equivalent

# Mixed-type comparisons are stricter in Python 3 than in Python 2
print(1 == 1.0)                  # True — numeric comparison crosses int/float
# print(1 < "1")                 # TypeError — won't compare int and str

# All-different chain
print(1 < 2 < 3 < 4)             # True

# Identity in a chain works too
y = None
print(y is None)                 # True

## Boolean operators and short-circuit evaluation

`and`, `or`, `not` are spelled out as words (no `&&` or `||`). Two important properties:

- **Short-circuit evaluation.** `a and b` doesn't evaluate `b` if `a` is falsy. `a or b` doesn't evaluate `b` if `a` is truthy.
- **Returns the *value*, not just a bool.** `"hello" and "world"` returns `"world"`, not `True`. `0 or "default"` returns `"default"`, not `True`. This is the basis of the classic `x = something or default` idiom.

For the strict bool form (`True`/`False`), use `bool(x)` explicitly, or the comparison operators (which always return real bools).

In [ ]:
# Short-circuit — divide-by-zero protected
n = 0
if n != 0 and (1 / n) > 0.5:     # 1/n is never evaluated because n != 0 is False
    print("big")
else:
    print("safe")

# Boolean operators return the actual operand, not a bool
print("hello" and "world")       # "world" — last truthy operand wins
print("" and "world")            # ""      — short-circuits on the empty string
print(None or "default")         # "default" — first truthy wins
print(0 or 0 or 0 or 42)         # 42

# To force a bool, wrap with bool()
print(bool("hello" and "world")) # True

## Identity vs equality — `is` vs `==`

The line to internalize:

- **`==`** — "do these objects compare equal?" Calls `__eq__`. The right tool for value comparison.
- **`is`** — "are these the same object?" Checks object identity. The right tool for sentinels (`is None`, `is True`).

Python *caches* small integers and short strings as a performance optimization — `5 is 5` happens to be `True`. **Do not rely on this.** It is an implementation detail. `1000 is 1000` may or may not be `True` depending on how the literals were parsed. The rule: use `==` for values, `is` only for `None` and the like.

In [ ]:
# Identity caching — implementation detail, don't rely on it
a = 5
b = 5
print(a is b)        # True — small ints are cached
print(a == b)        # True — equal values

c = 1000
d = 1000
print(c is d)        # could be True or False depending on context!
print(c == d)        # True — always

# The right rule
x = None
print(x is None)     # True
print(x == None)     # also True, but bad form

# For containers, `==` is element-wise equality
print([1, 2, 3] == [1, 2, 3])    # True
print([1, 2, 3] is [1, 2, 3])    # False — two different list objects

## Truthiness — what counts as false

Every Python object can be used in a boolean context (`if x`, `while x`, `bool(x)`). The default rule: an object is **truthy** unless it is one of these:

- `False`, `None`
- Numeric zero — `0`, `0.0`, `0j`, `Decimal(0)`
- Empty containers — `""`, `[]`, `()`, `{}`, `set()`
- Empty `bytes`/`bytearray` — `b""`
- Any object whose `__bool__` returns `False`, or whose `__len__` returns `0`

This rule is the basis of Python's most idiomatic "is there anything here?" check:

```python
if items:          # not `if len(items) > 0:`
    do_something()
```

The same trick fires for strings (`if name:`), dicts (`if config:`), and so on. It is the Python equivalent of "is it set, and not empty?" in one line.

In [ ]:
for value in [False, None, 0, 0.0, "", [], {}, (), set(), b"",
              True, 1, "x", [0], {"k": "v"}, (None,)]:
    print(f"{value!r:20s}  bool -> {bool(value)}")

## Mutable vs immutable — the property that explains aliasing

Every Python type is either **mutable** (can be changed in place after creation) or **immutable** (cannot be changed after creation).

**Immutable:** `int`, `float`, `bool`, `str`, `bytes`, `tuple`, `frozenset`, `None`.
**Mutable:** `list`, `dict`, `set`, `bytearray`, most user-defined classes.

This property explains every aliasing surprise. Two names pointing at the same *immutable* object can never go out of sync — neither can change it. Two names pointing at the same *mutable* object can, and you have to know they share state.

In [ ]:
# Immutable — no aliasing surprise possible
x = "hello"
y = x
y = y.upper()      # creates a new string, rebinds y
print(x)           # "hello" — unchanged
print(y)           # "HELLO"

# Mutable — aliasing makes mutation visible everywhere
xs = [1, 2, 3]
ys = xs
ys.append(99)      # mutates the shared list
print(xs)          # [1, 2, 3, 99] — surprise

# To get an independent copy, use .copy() or [:]
xs = [1, 2, 3]
ys = xs.copy()
ys.append(99)
print(xs)          # [1, 2, 3] — unchanged
print(ys)          # [1, 2, 3, 99]

# For deeply nested structures, the standard library has copy.deepcopy
import copy
nested = [[1, 2], [3, 4]]
shallow = nested.copy()
deep    = copy.deepcopy(nested)
shallow[0].append(99)    # mutates the inner list — shared with nested!
print(nested)            # [[1, 2, 99], [3, 4]]
print(deep)              # [[1, 2], [3, 4]] — truly independent

## Type conversion — explicit only

Python is **strongly typed**: it will not implicitly convert `int` to `str`, `str` to `int`, or `int` to `float` in most arithmetic operations. The conversions are explicit calls to the type:

```python
int("42")        # 42
int("0xff", 16)  # 255 — with explicit base
float("3.14")    # 3.14
str(42)          # "42"
bool(0)          # False
list("hello")    # ['h', 'e', 'l', 'l', 'o']
tuple([1, 2, 3]) # (1, 2, 3)
```

The exception that proves the rule: arithmetic between `int` and `float` produces a `float` — that one widening is automatic. Concatenation between `str` and `int` is *not* — `"value: " + 42` raises `TypeError`. Use `f"value: {n}"` instead.

In [ ]:
# Explicit conversions
print(int("42"))             # 42
print(int("0xff", 16))       # 255 — base argument
print(float("3.14e2"))       # 314.0
print(str(42))               # "42"
print(bool(""))              # False
print(list("hello"))         # ['h', 'e', 'l', 'l', 'o']

# What does NOT happen automatically
n = 42
# print("value: " + n)       # TypeError: can only concatenate str (not "int") to str
print(f"value: {n}")         # "value: 42"
print("value: " + str(n))    # also works, less readable

# Numeric widening IS automatic
print(1 + 2.0)               # 3.0 — int widens to float
print(type(1 + 2.0))         # <class 'float'>

## The walrus operator `:=`

Added in Python 3.8. Lets you **assign inside an expression** — combining "compute" and "name the result" in one step. Useful in `while` conditions and `if` checks where you'd otherwise compute the value twice or assign on a separate line.

The shape: `name := expression` returns the value of the expression AND binds it to `name`. Always written inside parentheses except in specific positions where the grammar allows it bare.

In [ ]:
# Without walrus — read each line, exit when we hit an empty one
lines = []
data = "first\nsecond\nthird\n\nfourth"

import io
f = io.StringIO(data)

# Pre-walrus version — assign once, check, repeat
while True:
    line = f.readline()
    if not line.strip():
        break
    lines.append(line.strip())
print(lines)

# With walrus — assign and check in one line
f = io.StringIO(data)
lines = []
while (line := f.readline().strip()):
    lines.append(line)
print(lines)

# In an if — also clean for length checks
data = "the quick brown fox jumps over the lazy dog"
if (n := len(data)) > 30:
    print(f"long string of {n} characters")

# In list comprehensions — defer expensive calls
import math
results = [y for x in range(5) if (y := math.sqrt(x * 10)) > 3]
print(results)

## Operator precedence — the table you bookmark

Python's precedence table, high to low. When in doubt, parenthesize. The compiler does not care; humans reading the code do.

| Tier | Operators |
|---|---|
| Highest | `(...)`, `[...]`, `{...}` — grouping/indexing |
| | `f(args)`, `x.attr`, `x[i]` |
| | `**` (right-associative) |
| | unary `+x`, `-x`, `~x` |
| | `*`, `/`, `//`, `%`, `@` |
| | `+`, `-` |
| | `<<`, `>>` |
| | `&` |
| | `^` |
| | `\|` |
| | comparisons: `==`, `!=`, `<`, `>`, `<=`, `>=`, `is`, `is not`, `in`, `not in` |
| | `not x` |
| | `and` |
| | `or` |
| Lowest | conditional expression: `x if cond else y` |

Three common pitfalls:

- `not x == y` parses as `not (x == y)`, not `(not x) == y` — usually what you wanted, but worth knowing.
- `**` is **right-associative**: `2 ** 3 ** 2 == 2 ** 9 == 512`, not `64`.
- Bitwise `&`, `|`, `^` bind *tighter* than comparison. `x == 1 & 2` is `x == (1 & 2)`, which is rarely what you meant.

## What's next

You now have Python's value semantics, the type system at the scalar level, the operators, the truthiness rule, and the mutable/immutable split that explains aliasing.

Notebook 03 puts these to work — control flow (`if`, `while`, `for`, `match`), and the function form that ties everything together. Default arguments, `*args` and `**kwargs`, keyword-only parameters, scoping rules, closures — the syntactic surface that makes Python feel light to write.